In [0]:
%sql
CREATE OR REPLACE TABLE workspace.fraud_lakehouse.gold_daily_fraud_summary
USING DELTA
CLUSTER BY (txn_date, channel)
AS
SELECT
    DATE(event_timestamp) AS txn_date,
    channel,
    location_country,
    COUNT(transaction_id) AS total_transactions,
    ROUND(SUM(amount_usd), 2) AS total_volume_usd,
    SUM(CASE WHEN risk_tier IN ('HIGH', 'CRITICAL') THEN 1 ELSE 0 END) AS flagged_fraud_count,
    ROUND(SUM(CASE WHEN risk_tier IN ('HIGH', 'CRITICAL') THEN amount_usd ELSE 0 END), 2) AS flagged_fraud_volume_usd,
    ROUND(
        (SUM(CASE WHEN risk_tier IN ('HIGH', 'CRITICAL') THEN 1 ELSE 0 END) * 100.0) / NULLIF(COUNT(transaction_id), 0),
        2
    ) AS fraud_rate_pct,
    SUM(CASE WHEN is_impossible_travel THEN 1 ELSE 0 END) AS impossible_travel_incidents,
    CURRENT_TIMESTAMP() AS _gold_updated_at
FROM workspace.fraud_lakehouse.silver_transactions
GROUP BY DATE(event_timestamp), channel, location_country;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.fraud_lakehouse.gold_high_risk_alerts
USING DELTA
CLUSTER BY (risk_tier, event_timestamp)
AS
SELECT
    t.transaction_id,
    t.account_id,
    t.user_id,
    p.primary_country AS registered_home_country,
    t.location_country AS transaction_country,
    t.channel,
    t.merchant_category,
    t.amount_usd,
    t.txns_last_1hr,
    t.spend_last_1hr_usd,
    t.is_impossible_travel,
    t.composite_risk_score,
    t.risk_tier,
    t.event_timestamp,
    CASE
        WHEN t.composite_risk_score >= 85 THEN 'AUTO_BLOCK_AND_ESCALATE'
        WHEN t.composite_risk_score >= 70 THEN 'IMMEDIATE_ANALYST_REVIEW'
        ELSE 'STEP_UP_2FA_CHALLENGE'
    END AS recommended_action,
    CURRENT_TIMESTAMP() AS _alert_generated_at
FROM workspace.fraud_lakehouse.silver_transactions t
LEFT JOIN workspace.fraud_lakehouse.silver_account_profiles p
    ON t.account_id = p.account_id
    AND p.is_current = true
WHERE t.risk_tier IN ('HIGH', 'CRITICAL');

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.fraud_lakehouse.gold_account_risk_exposure
USING DELTA
AS
SELECT
    p.account_id,
    p.user_id,
    p.risk_tier AS profile_risk_tier_at_txn,
    COUNT(t.transaction_id) AS txn_count_in_state,
    ROUND(SUM(t.amount_usd), 2) AS total_spend_usd,
    MAX(t.composite_risk_score) AS peak_txn_risk_score
FROM workspace.fraud_lakehouse.silver_account_profiles p
INNER JOIN workspace.fraud_lakehouse.silver_transactions t
    ON p.account_id = t.account_id
    AND t.event_timestamp >= p.effective_from
    AND (t.event_timestamp < p.effective_to OR p.effective_to IS NULL)
GROUP BY p.account_id, p.user_id, p.risk_tier;

In [0]:
%sql
SELECT * FROM workspace.fraud_lakehouse.gold_high_risk_alerts LIMIT 10;